<a href="https://colab.research.google.com/github/k1-r1ll/pytorch/blob/main/8_pytorch_back_propagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import torch
from random import randint

In [17]:
def act(z):
  return torch.tanh(z)

In [18]:
# производная функции активации: 1 - гиперболический тангенс в квадрате
def df(z):
  s = act(z)
  return 1 - s * s

In [19]:
def go_forward(x_inp, w1, w2):
  # Каждая строка матрицы весов — это набор весов для одного конкретного нейрона. torch.mv(w1[:, :3], x_inp) -- умножаем-суммируем первую строку на наши иксы и отдельно вторую
  # Если у автора кода матрица w1 изначально имела размер, например (2 Х 4) (где 2 строки и 4 столбца),
  # то выражение w1[:, :3] как раз отрежет матрицу размером (2 Х 3).
  # То есть веса берутся для обоих нейронов сразу (так как строк две), но отсекаются лишние столбцы.
  z1 = torch.mv(w1[:, :3], x_inp) + w1[:, 3] # получили вектор строго из двух значений (матрицу (2 Х 3) умножили на вектор (1 Х 3), после еще добавили два значения последней колонки
                                             # весов -- это смещение);
                                             # пайторч делает преобразоания
                                             # в плане строки-колонки после матрицы сам, мануально это правило тут не работает
  s1 = act(z1)  # Получили два значения сигмоиды
  z2 = torch.dot(w2[:2], s1) + w2[2]
  y = act(z2)   # Получили последнее значение сигмоилы

  return y, z1, s1, z2

In [20]:
torch.manual_seed(1)

In [21]:
W1 = torch.rand(8).view(2, 4) - 0.5
W2 = torch.rand(3) - 0.5

In [22]:
x_train = torch.FloatTensor([(-1, -1, -1), (-1, -1, 1), (-1, 1, -1), (-1, 1, 1),
                             (1, -1, -1),  (1, -1, 1),  (1, 1, -1),  (1, 1, 1)])
y_train = torch.FloatTensor([-1, 1, -1, 1, -1, 1, -1, -1])

In [23]:
lmd = 0.05
N = 1000
total = len(y_train)

In [24]:
for _ in range(N):
  k = randint(0, total-1)
  x = x_train[k]
  y, z1, s1, out = go_forward(x, W1, W2)
  e = y - y_train[k]
  delta = e * df(out) # out -- это z2: мы получаем по z2 сигмоиду и тут же считаем производную; локальный градиент 1
  delta2 = W2[:2] * delta * df(z1)  # локальный градиент 2: два веса для второго слоя умножаем на дельту выше и производную от двух зеток в конце первого слоя

  W2[:2] = W2[:2] - lmd * delta * s1  # шагаем назад для двух весов второго слоя (s1 -- две сигмоиды )
  W2[2] = W2[2] - lmd * delta # шагаем назад для единственного смещения второго слоя

  W1[0, :3] = W1[0, :3] - lmd * delta2[0] * x # шагаем назад для трех весов первого слоя (нейрон 1)
  W1[1, :3] = W1[1, :3] - lmd * delta2[1] * x # шагаем назад для трех весов первого слоя (нейрон 2)

  W1[0, 3] = W1[0, 3] - lmd * delta2[0] # шагаем назад для смещения первого слоя (нейрон 1)
  W1[1, 3] = W1[1, 3] - lmd * delta2[1] # шагаем назад для смещения первого слоя (нейрон 2)


In [25]:
for x, d in zip(x_train, y_train):
  y, z1, s1, out = go_forward(x, W1, W2)
  print(f"Выходное значение НС: {y} vs {d}")


Выходное значение НС: -0.9135374426841736 vs -1.0
Выходное значение НС: 0.989508867263794 vs 1.0
Выходное значение НС: -0.994165301322937 vs -1.0
Выходное значение НС: 0.9136945605278015 vs 1.0
Выходное значение НС: -0.9940142631530762 vs -1.0
Выходное значение НС: 0.9182131886482239 vs 1.0
Выходное значение НС: -0.9961453080177307 vs -1.0
Выходное значение НС: -0.8727749586105347 vs -1.0
